In [0]:
%python
'''
 CREATION OF DIMENSIONS --> Based on independent tables.
'''

'''
 Dimensional Modeling Rules 

    # 1. Surrogate Key (SK) -> Incremental or hash key for the Data Warehouse 
    # 2. Denormalize related data -> Join tables that strictly depend on a single core entity
    # 3. Select and Rename business fields
'''

##DIM_CUSTOMERS

In [0]:
%python
# 1. CUSTOMERS DIMENSION - Joining with CREDIT BUREAU to have complete customer info

## IMPORTS
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, concat_ws, sha2, coalesce, lit

'''
 1. Load tables from Silver Layer
'''
df_silver_customers = spark.table("banking_silver.customers")
df_silver_bureau = spark.table("banking_silver.credit_bureau_reports")


'''
 2. Select the most recent version of Credit Bureau per customer
'''
# If a customer has multiple reports, keep the latest (most recent) one

df_bureau_latest = (
    df_silver_bureau
    .orderBy(col("bureau_pull_date").desc())
    .dropDuplicates(["customer_id"])
)

'''
 3. Join information (LEFT JOIN to keep customers without bureau reports)
'''

df_dim_customers_joined = (
    df_silver_customers.alias("c")
    .join(
        df_bureau_latest.alias("b"),
        on=col("c.customer_id") == col("b.customer_id"),
        how="left"
    )
)

'''
 4. Dimensional Modeling: Generate Surrogate Key (SK) and select attributes
'''
df_dim_customers = (
    df_dim_customers_joined
    .withColumn("customer_sk", sha2(col("c.customer_id").cast("string"), 256)) # Surrogate Key for Gold joins - JOIN BRIDGE
    .withColumn("costumer_id_hash", sha2(col("c.customer_id").cast("string"), 256)) # Business ID Hash (PII protected) - Identical values here, but used for distinct business purposes.

    .select(
        # 1. Keys
        col("customer_sk"),
        col("costumer_id_hash"),
        
        # 2. Demographic Attributes and Segmentation (Excludes direct PII data like name/pan_number/email, etc.)
        col("date_of_birth"),
        col("kyc_status"),
        col("branch_code").alias("customer_branch_code"),
        col("is_legal"),
        
        # 3. Enriched Attributes (Silver Credit Bureau)
        coalesce(col("b.credit_score"), lit(0)).alias("credit_score"),
        coalesce(col("b.risk_grade"), lit("NO_REPORT")).alias("risk_grade"),
        coalesce(col("b.external_active_loans"), lit(0)).alias("external_active_loans"),
        coalesce(col("b.external_overdue_amount"), lit(0)).alias("external_overdue_amount"),
        col("b.bureau_pull_date").alias("last_bureau_pull_date"),
        
        # 4. Audit Metadata
        current_timestamp().alias("dim_created_at")
    )
)

'''
 5. Save Dimension to Gold Layer
'''
(
    df_dim_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.dim_customers")
)

display(spark.table("banking_gold.dim_customers"))

##DIM_ACCOUNTS

In [0]:
%python

## IMPORTS
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import col, current_timestamp, concat_ws, sha2, coalesce, lit

'''
 1. Load tables from Silver Layer
'''
df_silver_accounts = spark.table("banking_silver.accounts")

df_silver_accounts.printSchema()

df_silver_branches = spark.table("banking_silver.branches")

df_silver_branches.printSchema()

'''
 2. Join Accounts with Branches (LEFT JOIN)
'''
df_dim_accounts_joined = df_silver_accounts.alias("a").join(
    df_silver_branches.alias("b"), 
    on=col("a.branch_code") == col("b.branch_code"), 
    how="left"
)

'''
 3. Dimensional Modeling for Gold
'''

df_dim_accounts = df_dim_accounts_joined.select(
    # Clean Surrogate Key using direct SHA-256 on business ID
    sha2(col("a.account_id").cast("string"), 256).alias("account_sk"),
    
    # Foreign Key pointing to dim_customers (using direct SHA-256 on customer ID)
    sha2(col("a.customer_id").cast("string"), 256).alias("customer_sk"),
    
    # Account Attributes
    col("a.account_type"),
    col("a.balance"),
    col("a.currency"),
    col("a.status").alias("account_status"),
    col("a.opened_date"),
    
    # Branch Attributes
    col("a.branch_code"),
    coalesce(col("b.branch_name"), lit("NOT_PROVIDED")).alias("branch_name"),
    coalesce(col("b.city"), lit("NOT_PROVIDED")).alias("branch_city"),
    coalesce(col("b.state"), lit("NOT_PROVIDED")).alias("branch_state"),
    coalesce(col("b.region"), lit("NOT_PROVIDED")).alias("branch_region"),
    
    # Audit Metadata
    current_timestamp().alias("dim_created_at")
)

'''
 4. Save Dimension to Gold overwriting Schema
'''
(
    df_dim_accounts.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.dim_accounts")
)

display(spark.table("banking_gold.dim_accounts"))

#DIM_DATE

In [0]:
%python

## IMPORTS
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, explode, sequence, to_date, lit, 
    date_format, year, quarter, month, dayofmonth, 
    dayofweek, date_add, current_timestamp, weekofyear
)

'''
 1. Define the date range for the Data Warehouse (e.g. 10 years)
'''
start_date = "2020-01-01"
end_date = "2030-12-31"

'''
 2. Generate daily date sequence
'''
df_date_range = spark.range(1).select(
    explode(
        sequence(to_date(lit(start_date)), to_date(lit(end_date)), F.expr("interval 1 day"))
    ).alias("full_date")
)

'''
 3. Extract and build all temporal attributes for Gold
'''
df_dim_date = df_date_range.select(
    # Integer Surrogate Key (e.g.: 20261005)
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_id"),
    
    # Primary Date attribute
    col("full_date"),
    
    # Year, Quarter, Month, and Day components
    year(col("full_date")).alias("year"),
    quarter(col("full_date")).alias("quarter"),
    month(col("full_date")).alias("month"),
    dayofmonth(col("full_date")).alias("day_of_month"),
    
    # Week number of year (Spark direct native function)
    weekofyear(col("full_date")).alias("week_of_year"),
    
    # Descriptive names for reporting
    date_format(col("full_date"), "MMMM").alias("month_name"),
    date_format(col("full_date"), "MMM").alias("month_short"),
    date_format(col("full_date"), "EEEE").alias("day_name"),
    date_format(col("full_date"), "EEE").alias("day_short"),
    
    # Calendar / Week attributes
    dayofweek(col("full_date")).alias("day_of_week"),
    
    # Business Flags
    F.when(dayofweek(col("full_date")).isin([1, 7]), True).otherwise(False).alias("is_weekend"),
    F.when(col("full_date") == F.last_day(col("full_date")), True).otherwise(False).alias("is_month_end"),
    
    # Gold Ingestion Audit Metadata
    current_timestamp().alias("dim_created_at")
)

'''
 4. Save Dimension to Gold overwriting Schema
'''
(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.dim_date")
)

display(spark.table("banking_gold.dim_date"))

#FACT_TRANSACTIONS

In [0]:
%python

## IMPORTS
import pyspark.sql
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, current_timestamp, date_format, coalesce, lit, to_date, sha2
)

'''
 1. Load Silver tables and Gold Dimensions
'''
df_silver_tx = spark.table("banking_silver.transactions")
df_dim_accounts = spark.table("banking_gold.dim_accounts")
df_dim_customers = spark.table("banking_gold.dim_customers")

'''
 2. Join Transactions with dim_accounts to obtain account_sk and customer_id
'''
df_tx_acc = (
    df_silver_tx.alias("t")
    .join(
        df_dim_accounts.alias("a"),
        on=sha2(col("t.account_id").cast("string"), 256) == col("a.account_sk"),
        how="left"
    )
)

'''
 3. Join with dim_customers to obtain customer_sk
'''
df_tx_joined = (
    df_tx_acc.alias("ta")
    .join(
        df_dim_customers.alias("c"),
        on=col("ta.customer_sk") == col("c.customer_sk"),
        how="left"
    )
)

'''
 4. Selection and mapping mapped to actual Silver Transactions schema
'''
df_fact_transactions = df_tx_joined.select(
    # Natural Key / Business Key
    col("ta.txn_id").alias("transaction_id"),
    
    # Foreign Keys / Surrogate Keys pointing to Dimensions
    coalesce(col("ta.account_sk"), lit("UNKNOWN")).alias("account_sk"),
    coalesce(col("c.customer_sk"), lit("UNKNOWN")).alias("customer_sk"),
    date_format(to_date(col("ta.txn_timestamp")), "yyyyMMdd").cast("int").alias("transaction_date_id"),
    
    # Degenerate Attributes of Transaction
    col("ta.txn_type").alias("transaction_type"),
    col("ta.channel"),
    col("ta.status").alias("transaction_status"),
    
    # Quantitative Metrics (Facts)
    col("ta.amount"),
    
    # Transaction and Audit Timestamps
    col("ta.txn_timestamp"),
    current_timestamp().alias("fact_created_at")
)

'''
 5. Save Fact Table to Gold overwriting Schema
'''
(
    df_fact_transactions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("banking_gold.fact_transactions")
)

display(spark.table("banking_gold.fact_transactions"))